- **Goal:** Map JobTech occupation labels to SSYK codes
- **Challenges:** Different naming conventions, granularity mismatch
- **Decision:** Map at `occupation_group_label` level

## Inspect SSYK table

In [0]:
DESCRIBE silver_work_scb.ssyk_mapping;

In [0]:
SELECT *
FROM silver_work_scb.ssyk_mapping
LIMIT 5;

In [0]:
SELECT DISTINCT
    occupation_code,
    occupation_name
FROM silver_work_scb.ssyk_mapping
LIMIT 50;

## Inspect JobTech occupation labels

In [0]:
SELECT DISTINCT
    occupation_group_label,
    occupation_field_label,
    occupation_label
FROM silver_jobtech.job_postings_silver_clean
WHERE occupation_label IS NOT NULL
  AND TRIM(occupation_label) <> ''
LIMIT 50;

In [0]:
SELECT
    occupation_group_label,
    occupation_field_label,
    occupation_label,
    COUNT(*) AS posting_count
FROM silver_jobtech.job_postings_silver_clean
WHERE occupation_label IS NOT NULL
  AND TRIM(occupation_label) <> ''
GROUP BY
    occupation_group_label,
    occupation_field_label,
    occupation_label
ORDER BY posting_count DESC
LIMIT 50;

## Comparison
Based on the insights above we can not safely map `occupation_label` to `SSYK` directly because 
- JobTech data is messy, detailed, and sometimes multi-role: "Systemutvecklare/Programmerare", "Fordonsmekaniker/Bilmekaniker/Personbilsmekaniker"
- SSYK table is: structured, standardized, and hierarchical (codes like 1111, 0210, etc.)

This is a **many-to-many** + **fuzzy mapping** problem.

On the other hand `occupation_group_label` is MUCH closer to `SSYK` than `occupation_label` since it's either already standardized grouped or grouped: `Grundutbildade sjuksköterskor`, `Grundutbildade sjuksköterskor`, `Mjukvaru- och systemutvecklare m.fl.`

## Identify mapping strategy
So we are going to map at GROUP LEVEL using `occupation_group_label → SSYK`

This strategy is more stable since the group labels are consistent, the granularity matches since the SSYK and the group labels are already grouped.

But the SSYK is high level with limited subset
```
0110 Officerare
0210 Specialistofficerare
0310 Soldater m.fl.
1111 Politiker
```
So before mapping we want to check if `silver_work_scb.ssyk_mapping` complete because if it's too small we cannot use it directly.

In [0]:
SELECT DISTINCT occupation_group_label
FROM silver_jobtech.job_postings_silver_clean
WHERE occupation_group_label IS NOT NULL;

In [0]:
SELECT COUNT(*)
FROM silver_work_scb.ssyk_mapping;

In [0]:
SELECT *
FROM silver_work_scb.ssyk_mapping
LIMIT 50;

`432` rows strongly suggests this is a reasonably complete occupation reference table, not just a tiny sample. And from the values, many `occupation_group_label` values from `JobTech` appear to line up very closely with `occupation_name` in `silver_work_scb.ssyk_mapping`.

Howrver one important complication is that some SSYK names contain:

`nivå 1`, `nivå 2`, punctuation differences, and slight naming differences

So a simple exact join will work for some rows, but not all.

We likely need a two-step approach:

- Step 1: Try exact matching on cleaned text.
- Step 2: Handle remaining unmatched occupations manually or with rule-based logic.

In [0]:
SELECT
    COUNT(DISTINCT j.occupation_group_label) AS total_jobtech_groups,
    COUNT(DISTINCT s.occupation_name) AS matched_groups
FROM (
    SELECT DISTINCT occupation_group_label
    FROM silver_jobtech.job_postings_silver_clean
    WHERE occupation_group_label IS NOT NULL
) j
LEFT JOIN silver_work_scb.ssyk_mapping s
    ON TRIM(j.occupation_group_label) = TRIM(s.occupation_name);

In [0]:
SELECT
    j.occupation_group_label,
    s.occupation_code,
    s.occupation_name
FROM (
    SELECT DISTINCT occupation_group_label
    FROM silver_jobtech.job_postings_silver_clean
    WHERE occupation_group_label IS NOT NULL
) j
LEFT JOIN silver_work_scb.ssyk_mapping s
    ON TRIM(j.occupation_group_label) = TRIM(s.occupation_name)
WHERE s.occupation_code IS NOT NULL
ORDER BY j.occupation_group_label;

In [0]:
SELECT
    j.occupation_group_label
FROM (
    SELECT DISTINCT occupation_group_label
    FROM silver_jobtech.job_postings_silver_clean
    WHERE occupation_group_label IS NOT NULL
) j
LEFT JOIN silver_work_scb.ssyk_mapping s
    ON TRIM(j.occupation_group_label) = TRIM(s.occupation_name)
WHERE s.occupation_code IS NULL
ORDER BY j.occupation_group_label;

## Occupation mapping strategy

To harmonize JobTech occupation data with SCB labour-market datasets, the mapping will be performed at the `occupation_group_label` level rather than the `occupation_label` level.

### Why map at group level?
- `occupation_group_label` is more standardized and much closer to SSYK occupation names
- `occupation_label` is often too detailed, inconsistent, or multi-role
- group-level mapping is more stable and analytically defensible

### Initial mapping results
- Distinct JobTech occupation groups: **426**
- Exact SSYK matches found: **363**
- Exact match coverage: **~85%**

This confirms that `occupation_group_label` is a strong basis for harmonization.

### Remaining unmatched cases
The remaining unmatched occupation groups appear to be caused mainly by:
- spelling and abbreviation differences
- punctuation differences
- shortened forms such as `o` instead of `och`
- SSYK categories split into `nivå 1` and `nivå 2`

### Planned mapping approach
1. Exact match between `occupation_group_label` and `occupation_name`
2. Normalized text matching for minor naming differences
3. Manual override rules for ambiguous or level-split occupations

This approach provides a transparent and maintainable bridge between JobTech and SSYK, ready to support the Gold analytics layer.